# Tutorial of beam Uniform Load Capacity Calculator

#### Goal of the tool


The goal is to verify the structural integrety of the beams in a model, and locate the issues, so the engineer can remodel the blender script.

The following python script is able to withdraw the dimensions and materials of beams in an IFC file extracted from Blender. From which the script calculates the max line load capacity of every IFCbeam and identifies the beams below the required capacity, who are at risk of failure. 

The tool is made for the subject: STRUCTURE, and will be useful in the later stages of advanced building design, as it is simply a tool to check the different draft of the structural analysis.  

For the script to work you need a IFCfile of a blender model, that includes the structural design of the building. Furthermore you need the requried lineloads for each of your beamtypes.  

#### 1. A quick overview of the different scripts

In [1]:
import Dimensions
import GeoDimensions
import max_UDL

First step is to import ifcopenshell and the model we want to analyze

In [2]:
import ifcopenshell

model = ifcopenshell.open("samples/25-16-D-STR.ifc")

FileNotFoundError: File does not exist: 'samples\25-16-D-STR.ifc'.

##### Dimensions

The dimensions script (see file Dimensions.py) loads and reads the IFC file. It then uses Property sets to extract the name and dimensions of the beams. It print the GlobalID, Name, width (b), height (h), length (l), thickness falge (tf), thickness web (tw). The tf and tw values are only relevant for steel I-Beams.

In [12]:

DimensionsResult = Dimensions.beam_dimensions(model)
print("Beam Dimensions Result:", DimensionsResult)
print("Beam Dimensions Result:", DimensionsResult['1pwo0oySrANhXlWg9RRhly'])


NameError: name 'model' is not defined

#### Geometric dimensions 

If the beam dimensions is not inserted correctly as Property Sets in the IFCfile, the values can be found using the the ifcopenshell.geom import, which is the base for the script called GeoDimensions (see file GeoDimensions.py). Here the dimensions is calculated for each beam using its geometric measurements in the IFCfile from blender. 

In [ ]:
GeodimensionsResult =  GeoDimensions.checkRule(model)
print("GeoDimensions result:", GeodimensionsResult)
print(GeodimensionsResult['08KUHnYqn7HvCxSyjUsDVX'])

##### Max UDL

The results from the dimensions script is then loaded in to the Max_UDL script. 
This script uses the IFCBeam name to identify the material of the beam for example: 'Name': Structural Framing : Concrete - Rectangular (framing) gets sorted as concrete beams, and its dimension results data is used as input for the max line load capacity equation for concrete beams. There are a different equations for concrete (see Concrete_ULC.py), wood (see Glulam_ULC.py) and steel (see i_beam_ULC.py) beams and the output shows the line load capacity in the unit [kN/m]. 

In [ ]:
maxUDLResult = max_UDL.checkRule(DimensionsResult)
print("Max UDL result:", maxUDLResult)
print(maxUDLResult['08KUHnYqn7HvCxSyjUsDVX'])

Max UDL result: {'1Ab4ISo4v56xTNKdr$YdfU': (36.19435615204539, 'Glulam'), '1Ab4ISo4v56xTNKdr$Ydl2': (36.19435615204549, 'Glulam'), '1Ab4ISo4v56xTNKdr$Ydm8': (36.19435615204539, 'Glulam'), '1Ab4ISo4v56xTNKdr$Ydoy': (36.19435615204539, 'Glulam'), '1Ab4ISo4v56xTNKdr$Ydnc': (36.19435615204549, 'Glulam'), '1Ab4ISo4v56xTNKdr$YdqC': (36.19435615204525, 'Glulam'), '1Ab4ISo4v56xTNKdr$YdnU': (36.19435615204539, 'Glulam'), '1Ab4ISo4v56xTNKdr$Ydrg': (36.19435615204549, 'Glulam'), '1Ab4ISo4v56xTNKdr$Ydum': (36.19435615204549, 'Glulam'), '1Ab4ISo4v56xTNKdr$YdxI': (36.19435615204549, 'Glulam'), '1Ab4ISo4v56xTNKdr$YdvL': (36.19435615204549, 'Glulam'), '1Ab4ISo4v56xTNKdr$YdzN': (36.19435615204549, 'Glulam'), '1Ab4ISo4v56xTNKdr$Yd$P': (36.19435615204525, 'Glulam'), '1Ab4ISo4v56xTNKdr$Yd2n': (386.8632657757559, 'Glulam'), '1Ab4ISo4v56xTNKdr$Yd6m': (386.86326577573436, 'Glulam'), '1Ab4ISo4v56xTNKdr$Yd4W': (36.19435615204525, 'Glulam'), '1Ab4ISo4v56xTNKdr$Yd4O': (36.19435615204539, 'Glulam'), '1Ab4ISo4v56x

#### Final Output

The following script prints the final output. Our goal was to identify the beams with too low capacity, who are at risk of failure. This script inputs the design line load along side the max UDL results from before and tests the two values. In this example the line loads are pulled from the structural report of the model. 

The scripts prints the amount of correctly dimensioned beams, as well as the amount where the strength is BELOW the required capacity. This is done for each material group. 

The GlobalID's of the IFCbeams below the required capacity are printed, so they can be located in the model and redimensioned.  

In [ ]:
LineloadsReport = {"Concrete": 26.12, "Steel": 14.97, "Glulam": 17.2}

# Validate beams against LineloadsReport
material_counts_right = {"Concrete": 0, "Steel": 0, "Glulam": 0}
material_counts_wrong = {"Concrete": 0, "Steel": 0, "Glulam": 0}
beams_not_dimensioned_right = {"Concrete": [], "Steel": [], "Glulam": []}

for beam_id, result in maxUDLResult.items():
    if result is None:
        continue
    
    max_udl, material_type = result
    
    # Normalize material type for lookup
    material_key = material_type.capitalize()
    
    if material_key in LineloadsReport:
        allowable_load = LineloadsReport[material_key]
        
        # Check if max_udl is higher than or equal to the allowable load
        if max_udl >= allowable_load:
            material_counts_right[material_key] += 1
        else:
            material_counts_wrong[material_key] += 1
            # beam_id is the GlobalId, so store it directly
            beams_not_dimensioned_right[material_key].append(beam_id)
        
# Print results
print(f"{material_counts_right['Concrete']} Concrete beams ABOVE required capacity")
print(f"{material_counts_wrong['Concrete']} Concrete beams BELOW required capacity")
print(f"{material_counts_right['Steel']} Steel beams ABOVE required capacity")
print(f"{material_counts_wrong['Steel']} Steel beams BELOW required capacity")
print(f"{material_counts_right['Glulam']} Glulam beams ABOVE required capacity")
print(f"{material_counts_wrong['Glulam']} Glulam beams BELOW required capacity")

# Print GlobalIds of all beams that are NOT dimensioned right
print("\n--- Beams that arent strong enough ---")
for material, globalids in beams_not_dimensioned_right.items():
    if globalids:
        print(f"\n{material} ({len(globalids)}):")
        for globalid in globalids:
            print(f"  {globalid}")

      

124 Concrete beams ABOVE required capacity
9 Concrete beams BELOW required capacity
11 Steel beams ABOVE required capacity
0 Steel beams BELOW required capacity
148 Glulam beams ABOVE required capacity
35 Glulam beams BELOW required capacity

--- Beams that arent strong enough ---

Concrete (9):
  0GlvZwwxr4yeet926EnPJ2
  1gBbes5lD1DQsWtn$7fdgy
  1IYmUvZl10xhDtPsDD62My
  1IYmUvZl10xhDtPsDD62et
  1IYmUvZl10xhDtPsDD62dc
  1IYmUvZl10xhDtPsDD62dk
  1IYmUvZl10xhDtPsDD62w6
  1NYSK5JsH9cfzXMSFQZbo8
  1NYSK5JsH9cfzXMSFQZbFA

Glulam (35):
  1Ab4ISo4v56xTNKdr$YP5U
  1Ab4ISo4v56xTNKdr$YP9W
  1Ab4ISo4v56xTNKdr$YPAv
  1Ab4ISo4v56xTNKdr$YPCu
  2MZkIr_HjEQ9IL5AB3PY3i
  2MZkIr_HjEQ9IL5AB3PY3W
  2MZkIr_HjEQ9IL5AB3PY3a
  2MZkIr_HjEQ9IL5AB3PY2J
  2MZkIr_HjEQ9IL5AB3PY2c
  2MZkIr_HjEQ9IL5AB3PYT9
  2MZkIr_HjEQ9IL5AB3PYTS
  2MZkIr_HjEQ9IL5AB3PYTl
  2MZkIr_HjEQ9IL5AB3PYTo
  2MZkIr_HjEQ9IL5AB3PYSX
  2MZkIr_HjEQ9IL5AB3PYSq
  2MZkIr_HjEQ9IL5AB3PYV7
  2MZkIr_HjEQ9IL5AB3PYVg
  2MZkIr_HjEQ9IL5AB3PYVz
  2MZkIr_HjEQ9